<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/week2_day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# STEP 1: Install dependencies
# =========================================================
!pip install -q sentence-transformers faiss-cpu openai scikit-learn


# =========================================================
# STEP 2: Imports
# =========================================================
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from google.colab import userdata


# =========================================================
# STEP 3: Initialize OpenAI
# =========================================================
os.environ["OPENAI_API_KEY"] = userdata.get("WEEK2_KEY")
client = OpenAI()


# =========================================================
# STEP 4: Dataset
# =========================================================
dataset = """
Artificial intelligence refers to the simulation of human intelligence in machines.
Machine learning is a subset of artificial intelligence.
Deep learning uses neural networks with multiple layers.
Large language models are trained on massive text datasets.
Embeddings are numerical vector representations of text.
Vector databases store embeddings for fast retrieval.
Similarity search finds nearest vectors to a query.
FAISS is a library developed by Facebook AI Research.
FAISS supports fast vector similarity search.
Retrieval Augmented Generation combines retrieval and generation.
RAG improves factual correctness of language models.

The system uses IndexFlatL2 internally.
Developers prefer cosine similarity in many systems.
FAISS supports IndexFlatL2 for vector comparison.
Indexes can be flat or inverted.
"""


# =========================================================
# STEP 5: Chunking
# =========================================================
chunks = [line.strip() for line in dataset.split("\n") if line.strip()]
print("Total chunks:", len(chunks))


# =========================================================
# STEP 6: Embeddings + FAISS
# =========================================================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(chunks).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)


# =========================================================
# STEP 7: Keyword index (TF-IDF)
# =========================================================
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(chunks)


# =========================================================
# STEP 8: Retrieval Methods
# =========================================================
def vector_retrieve(query, top_k=3):
    q_emb = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    scores = 1 / (1 + distances[0])
    return indices[0], scores


def keyword_retrieve(query, top_k=3):
    q_vec = tfidf.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices, scores[top_indices]


def hybrid_retrieve(query, top_k=3, alpha=0.5):
    v_idx, v_scores = vector_retrieve(query, len(chunks))
    k_idx, k_scores = keyword_retrieve(query, len(chunks))

    hybrid_scores = {}
    for i in range(len(chunks)):
        hybrid_scores[i] = (
            alpha * v_scores[v_idx.tolist().index(i)]
            + (1 - alpha) * k_scores[k_idx.tolist().index(i)]
        )

    ranked = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    final_indices = [idx for idx, _ in ranked[:top_k]]
    final_scores = [score for _, score in ranked[:top_k]]

    return final_indices, final_scores


# =========================================================
# STEP 9: RAG Answer Generation
# =========================================================
def generate_answer(question, passages):
    context = "\n".join(passages)
    prompt = f"""
Use only the context below to answer.

Context:
{context}

Question:
{question}

Answer:
"""
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt,
        temperature=0
    )
    return response.output_text


# =========================================================
# STEP 10: Interactive Comparison Chat
# =========================================================
print("Hybrid RAG Chat Ready ✅")
print("Type 'exit' to quit\n")

while True:
    query = input("You: ")

    if query.lower() in ["exit", "quit"]:
        print("Chat ended ✅")
        break

    v_idx, v_scores = vector_retrieve(query)
    v_passages = [chunks[i] for i in v_idx]
    v_answer = generate_answer(query, v_passages)

    h_idx, h_scores = hybrid_retrieve(query)
    h_passages = [chunks[i] for i in h_idx]
    h_answer = generate_answer(query, h_passages)

    print("\n--- Vector Retrieval ---")
    for i, s in zip(v_idx, v_scores):
        print(f"[Score: {s:.3f}] {chunks[i]}")

    print("\nAnswer (Vector):", v_answer)

    print("\n--- Hybrid Retrieval ---")
    for i, s in zip(h_idx, h_scores):
        print(f"[Score: {s:.3f}] {chunks[i]}")

    print("\nAnswer (Hybrid):", h_answer)
    print("=" * 60)